In [48]:
"""Pipeline ETL – Base Olist → SQL Server"""
from __future__ import annotations

import logging, re, time, unicodedata
from datetime import datetime
from pathlib import Path

import pandas as pd
import sqlalchemy as sa


In [49]:
#Configurações do pipeline.
PASTA_DADOS  = Path(r"C:\Users\Carla\Downloads\BaseOlist")
DIR_PIPELINE = Path("pipeline")
LOCK_FILE    = DIR_PIPELINE / "pipeline.lock"
LOG_PATH     = DIR_PIPELINE / "logs" / "pipeline.log"

DATABASE_URL = (
    "mssql+pyodbc://@localhost/Olist_Data"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes"
)

COLUNAS_OBRIGATORIAS = {
    "olist_orders_dataset":    ["order_id", "customer_id"],
    "olist_customers_dataset": ["customer_id"],
} 

In [50]:
#Preparação de ambiente.
for _d in (DIR_PIPELINE / "logs", DIR_PIPELINE / "rejeitados"):
    _d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=LOG_PATH, level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s", encoding="utf-8"
)

engine = sa.create_engine(DATABASE_URL, fast_executemany=True, pool_pre_ping=True)

In [51]:
#Padronização 
def normalizar(txt: str) -> str:
    sem_acento = unicodedata.normalize("NFKD", txt).encode("ASCII", "ignore").decode()
    return re.sub(r"[^a-z0-9_]", "", re.sub(r"\s+", "_", sem_acento.lower().strip()))


def _limpar_sql(sql: str) -> str:
    """Remove comentários de linha (--) e bloco (/* */) sem tocar no conteúdo."""
    sql = re.sub(r"/\*.*?\*/", "", sql, flags=re.DOTALL)   # /* bloco */
    sql = re.sub(r"--[^\n]*", "", sql)                      # -- linha
    return sql


def _split_por_go(sql: str) -> list[str]:
    """Divide o SQL em statements usando GO como separador de batch."""
    partes = re.split(r"^\s*GO\s*$", sql, flags=re.MULTILINE | re.IGNORECASE)
    return [s.strip() for s in partes if s.strip()]


In [52]:
#Estrutura para log.
def criar_tabela_auditoria() -> None:
    with engine.begin() as conn:
        conn.execute(sa.text("""
            IF NOT EXISTS (SELECT 1 FROM sysobjects WHERE name='etl_log' AND xtype='U')
            CREATE TABLE etl_log (
                id          BIGINT IDENTITY PRIMARY KEY,
                arquivo     VARCHAR(255),
                tabela      VARCHAR(255),
                linhas      INT,
                invalidas   INT,
                duplicadas  INT,
                duracao_seg DECIMAL(10,2),
                status      VARCHAR(20),
                erro        VARCHAR(MAX),
                data_carga  DATETIME2 DEFAULT GETDATE()
            )
        """))

def registrar_log(reg: dict) -> None:
    with engine.begin() as conn:
        conn.execute(sa.text("""
            INSERT INTO etl_log (arquivo, tabela, linhas, invalidas, duplicadas, duracao_seg, status, erro)
            VALUES (:arquivo, :tabela, :linhas, :invalidas, :duplicadas, :duracao_seg, :status, :erro)
        """), reg)


In [53]:
#Processamento(Limpeza e deduplicação).
def processar(arquivo: Path) -> None:
    tabela = normalizar(arquivo.stem)
    t0     = time.perf_counter()
    inv    = pd.DataFrame()

    try:
        df = pd.read_csv(arquivo, encoding="latin_1")
        df.columns = [normalizar(c) for c in df.columns]
        df.dropna(how="all", inplace=True)

        obrig = COLUNAS_OBRIGATORIAS.get(tabela, [])
        inv   = df[df[obrig].isnull().any(axis=1)] if obrig else pd.DataFrame()
        df    = df.drop(inv.index)

        if not inv.empty:
            inv.to_csv(DIR_PIPELINE / "rejeitados" / f"{tabela}.csv", index=False)

        subset = ["order_id"] if "order_id" in df.columns else None
        antes  = len(df)
        df.drop_duplicates(subset=subset, inplace=True)
        duplicadas = antes - len(df)

        df["data_carga"] = datetime.now()

        with engine.begin() as conn:
            df.to_sql(tabela, conn, if_exists="append", index=False, method=None, chunksize=500)

        duracao = round(time.perf_counter() - t0, 2)
        logging.info("CARGA OK | %s | linhas=%d | inv=%d | dup=%d | %.2fs",
                     arquivo.name, len(df), len(inv), duplicadas, duracao)
        print(f"  OK  {arquivo.name:<50} {len(df):>6} linhas | {duracao}s")

        registrar_log({"arquivo": arquivo.name, "tabela": tabela, "linhas": len(df),
                       "invalidas": len(inv), "duplicadas": duplicadas,
                       "duracao_seg": duracao, "status": "OK", "erro": None})

    except Exception as exc:
        duracao = round(time.perf_counter() - t0, 2)
        logging.error("CARGA ERRO | %s | %s", arquivo.name, exc)
        print(f" ERRO  {arquivo.name}: {exc}")
        registrar_log({"arquivo": arquivo.name, "tabela": tabela, "linhas": 0,
                       "invalidas": len(inv), "duplicadas": 0,
                       "duracao_seg": duracao, "status": "ERRO", "erro": str(exc)})

In [54]:

# ── Modelagem ─────────────────────────────────────────────
def executar_modelagem() -> None:
    print("\n" + "="*60)
    print("INICIANDO MODELAGEM")
    print("="*60)

    if not SQL_MODELAGEM.exists():
        msg = f"Arquivo não encontrado: {SQL_MODELAGEM}"
        logging.error("MODELAGEM | %s", msg)
        print(f" ERRO  {msg}")
        registrar_log({"arquivo": str(SQL_MODELAGEM), "tabela": "modelagem",
                       "linhas": 0, "invalidas": 0, "duplicadas": 0,
                       "duracao_seg": 0, "status": "ERRO", "erro": msg})
        return

    sql_bruto  = SQL_MODELAGEM.read_text(encoding="utf-8")
    sql_limpo  = _limpar_sql(sql_bruto)
    statements = _split_por_go(sql_limpo)
    total      = len(statements)

    logging.info("MODELAGEM | arquivo lido | %d statements encontrados", total)
    print(f"  Arquivo : {SQL_MODELAGEM.name}")
    print(f"  Statements encontrados: {total}\n")

    erros = 0
    for i, stmt in enumerate(statements, 1):
        preview  = " ".join(stmt.split())[:80]   # preview compacto sem quebras
        t0       = time.perf_counter()
        try:
            with engine.begin() as conn:
                conn.execute(sa.text(stmt))
            duracao = round(time.perf_counter() - t0, 2)
            logging.info("MODELAGEM [%d/%d] OK | %.2fs | %s", i, total, duracao, preview)
            print(f"  [{i:>2}/{total}] OK    {duracao:>5.2f}s — {preview[:60]}...")
            registrar_log({"arquivo": SQL_MODELAGEM.name, "tabela": f"modelagem_stmt_{i:03d}",
                           "linhas": 0, "invalidas": 0, "duplicadas": 0,
                           "duracao_seg": duracao, "status": "OK", "erro": None})
        except Exception as exc:
            duracao = round(time.perf_counter() - t0, 2)
            erros  += 1
            logging.error("MODELAGEM [%d/%d] ERRO | %s | stmt: %s", i, total, exc, preview)
            print(f"  [{i:>2}/{total}] ERRO  {duracao:>5.2f}s — {preview[:60]}")
            print(f"           Detalhe: {exc}\n")
            registrar_log({"arquivo": SQL_MODELAGEM.name, "tabela": f"modelagem_stmt_{i:03d}",
                           "linhas": 0, "invalidas": 0, "duplicadas": 0,
                           "duracao_seg": duracao, "status": "ERRO", "erro": str(exc)})
            # Continua nos próximos statements para auditar tudo, não interrompe

    print("="*60)
    if erros == 0:
        logging.info("MODELAGEM CONCLUÍDA | %d/%d statements OK", total, total)
        print(f"  MODELAGEM CONCLUÍDA — {total}/{total} statements OK")
    else:
        logging.warning("MODELAGEM COM ERROS | %d erros em %d statements", erros, total)
        print(f"  MODELAGEM COM ERROS — {erros} erro(s) em {total} statements")
        print(f"  Consulte etl_log (tabela='modelagem_stmt_*') para detalhes.")
    print("="*60 + "\n")


In [55]:
#Main
def executar() -> None:
    if LOCK_FILE.exists():
        raise RuntimeError("Pipeline já em execução. Remova pipeline.lock se necessário.")
    LOCK_FILE.touch()
    try:
        criar_tabela_auditoria()
        arquivos = sorted(PASTA_DADOS.glob("*.csv"))
        if not arquivos:
            print(f"Nenhum CSV encontrado em: {PASTA_DADOS}")
            return
        print(f"{len(arquivos)} arquivo(s) encontrado(s)\n")
        for arquivo in arquivos:
            processar(arquivo)
        executar_modelagem()
    finally:
        LOCK_FILE.unlink(missing_ok=True)

if __name__ == "__main__":
    executar()

8 arquivo(s) encontrado(s)

  OK  olist_customers_dataset.csv                         99441 linhas | 5.82s
  OK  olist_geolocation_dataset.csv                      738327 linhas | 37.24s
  OK  olist_order_items_dataset.csv                       98666 linhas | 5.37s
  OK  olist_order_payments_dataset.csv                    99440 linhas | 4.71s
  OK  olist_order_reviews_dataset.csv                     98673 linhas | 6.72s
  OK  olist_orders_dataset.csv                            99441 linhas | 7.46s
  OK  olist_products_dataset.csv                          32951 linhas | 1.72s
  OK  olist_sellers_dataset.csv                            3095 linhas | 0.15s

INICIANDO MODELAGEM
  Arquivo : Olistranformacao_modelagem.sql
  Statements encontrados: 42

  [ 1/42] OK     0.00s — DROP TABLE IF EXISTS dbo.olist_customers_stg;...
  [ 2/42] OK     0.24s — SELECT TRY_CAST(customer_id AS VARCHAR(32)) AS customer_id, ...
  [ 3/42] OK     0.00s — DROP TABLE IF EXISTS dbo.olist_products_stg;...
  [ 4/42]